In [1]:
import pandas as pd

In [2]:
file_path = "../data/raw/betavanx_mvp_v1.xlsx"

In [3]:
xls = pd.ExcelFile(file_path)

xls.sheet_names

['Project', 'Task', 'Resource', 'DailyReport', 'ResourceUsage']

In [4]:
project_df = pd.read_excel(xls, sheet_name="Project")
task_df = pd.read_excel(xls, sheet_name="Task")
resource_df = pd.read_excel(xls, sheet_name="Resource")
daily_df = pd.read_excel(xls, sheet_name="DailyReport")
usage_df = pd.read_excel(xls, sheet_name="ResourceUsage")

In [5]:
merged_df = daily_df.merge(task_df, left_on="task_id", right_on="id")
merged_df.head()

,id_x,project_id_x,task_id,date,progress_percent,actual_cost,id_y,project_id_y,name,planned_duration,planned_cost
0,1,1,1,2026-01-01,10,50000,1,1,Excavation,5,300000
1,2,1,1,2026-01-02,25,120000,1,1,Excavation,5,300000
2,3,1,1,2026-01-03,40,200000,1,1,Excavation,5,300000
3,4,1,2,2026-01-06,10,100000,2,1,Foundation,10,800000
4,5,1,2,2026-01-07,20,250000,2,1,Foundation,10,800000


In [6]:
progress_summary = merged_df.groupby("task_id").agg({
    "progress_percent": "max",
    "actual_cost": "sum",
    "planned_cost": "first"
}).reset_index()
progress_summary 

,task_id,progress_percent,actual_cost,planned_cost
0,1,40,370000,300000
1,2,20,350000,800000


In [7]:
progress_summary["expected_cost"] = (
    progress_summary["planned_cost"] * 
    (progress_summary["progress_percent"] / 100)
)

In [8]:
def check_status(row):
    expected_cost = row["planned_cost"] * (row["progress_percent"] / 100)

    if row["actual_cost"] > expected_cost:
        return "⚠️ Over Budget vs Progress"
    elif row["progress_percent"] == 100:
        return "✅ Completed"
    else:
        return "OK"
progress_summary["status"] = progress_summary.apply(check_status, axis=1)

In [9]:
progress_summary

,task_id,progress_percent,actual_cost,planned_cost,expected_cost,status
0,1,40,370000,300000,120000.0,⚠️ Over Budget vs Progress
1,2,20,350000,800000,160000.0,⚠️ Over Budget vs Progress


In [10]:
progress_summary

,task_id,progress_percent,actual_cost,planned_cost,expected_cost,status
0,1,40,370000,300000,120000.0,⚠️ Over Budget vs Progress
1,2,20,350000,800000,160000.0,⚠️ Over Budget vs Progress


In [11]:
progress_summary["cost_variance"] = (
    progress_summary["actual_cost"] - progress_summary["expected_cost"]
)
progress_summary["cost_variance_ratio"] = (
    progress_summary["cost_variance"] / progress_summary["expected_cost"]
)

progress_summary

,task_id,progress_percent,actual_cost,planned_cost,expected_cost,status,cost_variance,cost_variance_ratio
0,1,40,370000,300000,120000.0,⚠️ Over Budget vs Progress,250000.0,2.083333
1,2,20,350000,800000,160000.0,⚠️ Over Budget vs Progress,190000.0,1.187500


In [12]:
def calculate_score(row):
    ratio = row["cost_variance_ratio"]

    if pd.isna(ratio):
        return None
    
    if ratio <= 0:
        return 100  # خوب یا بهینه
    
    elif ratio <= 0.2:
        return 70   # کمی بد
    
    elif ratio <= 0.5:
        return 40   # بد
    
    else:
        return 10   # خیلی بد

In [13]:
progress_summary["score"] = progress_summary.apply(calculate_score, axis=1)

In [14]:
progress_summary

,task_id,progress_percent,actual_cost,planned_cost,expected_cost,status,cost_variance,cost_variance_ratio,score
0,1,40,370000,300000,120000.0,⚠️ Over Budget vs Progress,250000.0,2.083333,10
1,2,20,350000,800000,160000.0,⚠️ Over Budget vs Progress,190000.0,1.187500,10


In [16]:
daily_df["date"] = pd.to_datetime(daily_df["date"])

In [17]:
schedule_df = daily_df.groupby("task_id").agg({
    "date": ["min", "max"]
}).reset_index()

In [18]:
schedule_df.columns = ["task_id", "start_date", "current_date"]

In [19]:
schedule_df["days_passed"] = (
    schedule_df["current_date"] - schedule_df["start_date"]
).dt.days + 1

In [20]:
schedule_df = schedule_df.merge(
    task_df[["id", "planned_duration"]],
    left_on="task_id",
    right_on="id",
    how="left"
)

In [21]:
schedule_df["planned_progress"] = (
    schedule_df["days_passed"] / schedule_df["planned_duration"]
) * 100

In [22]:
progress_summary = progress_summary.merge(
    schedule_df[["task_id", "planned_progress"]],
    on="task_id",
    how="left"
)

In [23]:
progress_summary["schedule_variance"] = (
    progress_summary["progress_percent"] - progress_summary["planned_progress"]
)

In [24]:
progress_summary

,task_id,progress_percent,actual_cost,planned_cost,expected_cost,status,cost_variance,cost_variance_ratio,score,planned_progress,schedule_variance
0,1,40,370000,300000,120000.0,⚠️ Over Budget vs Progress,250000.0,2.083333,10,60.0,-20.0
1,2,20,350000,800000,160000.0,⚠️ Over Budget vs Progress,190000.0,1.187500,10,20.0,0.0


In [25]:
def cost_score(row):
    if row["expected_cost"] == 0:
        return 0

    deviation_ratio = abs(row["cost_variance"]) / row["expected_cost"]
    score = 100 - (deviation_ratio * 100)

    return max(score, 0)

In [26]:
def schedule_score(row):
    if row["planned_progress"] == 0:
        return 0

    delay_ratio = abs(row["schedule_variance"]) / row["planned_progress"]
    score = 100 - (delay_ratio * 100)

    return max(score, 0)

In [27]:
progress_summary["cost_score"] = progress_summary.apply(cost_score, axis=1)
progress_summary["schedule_score"] = progress_summary.apply(schedule_score, axis=1)

In [28]:
progress_summary["final_score"] = (
    0.6 * progress_summary["cost_score"] +
    0.4 * progress_summary["schedule_score"]
)

In [29]:
progress_summary

,task_id,progress_percent,actual_cost,planned_cost,expected_cost,status,cost_variance,cost_variance_ratio,score,planned_progress,schedule_variance,cost_score,schedule_score,final_score
0,1,40,370000,300000,120000.0,⚠️ Over Budget vs Progress,250000.0,2.083333,10,60.0,-20.0,0,66.666667,26.666667
1,2,20,350000,800000,160000.0,⚠️ Over Budget vs Progress,190000.0,1.187500,10,20.0,0.0,0,100.000000,40.000000


In [30]:
def generate_alert(row):
    if row["final_score"] < 50:
        return "🔴 Critical"
    elif row["cost_variance"] > 0 and row["schedule_variance"] < 0:
        return "🔴 High Risk"
    elif row["final_score"] < 75:
        return "🟡 Warning"
    else:
        return "🟢 Good"
progress_summary["alert"] = progress_summary.apply(generate_alert, axis=1)
progress_summary

,task_id,progress_percent,actual_cost,planned_cost,expected_cost,status,cost_variance,cost_variance_ratio,score,planned_progress,schedule_variance,cost_score,schedule_score,final_score,alert
0,1,40,370000,300000,120000.0,⚠️ Over Budget vs Progress,250000.0,2.083333,10,60.0,-20.0,0,66.666667,26.666667,🔴 Critical
1,2,20,350000,800000,160000.0,⚠️ Over Budget vs Progress,190000.0,1.187500,10,20.0,0.0,0,100.000000,40.000000,🔴 Critical


In [31]:
project_score = (
    progress_summary["final_score"] * progress_summary["planned_cost"]
).sum() / progress_summary["planned_cost"].sum()

In [32]:
project_cost_variance = progress_summary["cost_variance"].sum()

In [33]:
project_schedule_variance = progress_summary["schedule_variance"].mean()

In [34]:
def project_alert(score):
    if score < 50:
        return "🔴 Critical"
    elif score < 75:
        return "🟡 Warning"
    else:
        return "🟢 Good"

In [35]:
project_alert_status = project_alert(project_score)

In [36]:
print("Project Score:", project_score)
print("Cost Variance:", project_cost_variance)
print("Schedule Variance:", project_schedule_variance)
print("Alert:", project_alert_status)

Project Score: 36.36363636363637
Cost Variance: 440000.0
Schedule Variance: -10.0
Alert: 🔴 Critical


In [37]:
dashboard_df = progress_summary[
    [
        "task_id",
        "progress_percent",
        "planned_progress",
        "cost_variance",
        "schedule_variance",
        "final_score",
        "alert"
    ]
]

In [38]:
dashboard_df = dashboard_df.round({
    "progress_percent": 1,
    "planned_progress": 1,
    "cost_variance": 0,
    "schedule_variance": 1,
    "final_score": 1
})


In [39]:
dashboard_df = dashboard_df.sort_values(by="final_score")

In [40]:
dashboard_df.style.format({
    "cost_variance": "{:,.0f}"
})

,task_id,progress_percent,planned_progress,cost_variance,schedule_variance,final_score,alert
0,1,40,60.000000,"250,000",-20.000000,26.700000,🔴 Critical
1,2,20,20.000000,"190,000",0.000000,40.000000,🔴 Critical


In [41]:
dashboard_df

,task_id,progress_percent,planned_progress,cost_variance,schedule_variance,final_score,alert
0,1,40,60.0,250000.0,-20.0,26.7,🔴 Critical
1,2,20,20.0,190000.0,0.0,40.0,🔴 Critical


In [44]:
dashboard_df.to_json("../frontend/app/data/dashboard.json", orient="records")